# 08 – NLP Analysis

### Purpose of the Notebook
Analyse textbasierter Vergabefelder mittels NLP.

### Steps
- TF‑IDF preprocessing
- SVD dimensionality reduction
- NMF topic modelling
- SVM text‑risk classifier (3 classes)
- Export TEXT_RISK_SCORE + NLP features
- Integration into modelling pipeline

--------------------
#### Imports & Setup & Dataset
-------------------

In [38]:
# ---------------------------------------------------------
# Import moduls
# ---------------------------------------------------------


# import standard modules
import pandas as pd
import numpy as np
from pathlib import Path

import sys
from pathlib import Path

In [39]:
# shut off some annoying warnings
import warnings

warnings.filterwarnings("ignore", message="A value is trying to be set on a copy")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [40]:
# ---------------------------------------------------------
# Setup style
# ---------------------------------------------------------

# show all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)

In [46]:
# ---------------------------------------------------------
# Load scripts
# ----------------------------------------------------------

%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

# found min directory
PROJECT_ROOT = Path("..").resolve()

# maindirectory sys.path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# import function from script
from my_scripts.nlp_processing import (text_preproceccing, build_tfidf_svd, build_nmf_topics,
                               build_text_risk_classifier, predict_text_risk)

from my_scripts.eda import (overview, filter_germany)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
# ---------------------------------------------------------
# Import dataset
# ---------------------------------------------------------

df = pd.read_pickle("../data/dataset_comp.pkl")

print("EU dataset:", df.shape)

EU dataset: (4039906, 39)


----------------
### NLP PROCESSING

----------

In [48]:
# ---------------------------------------------------------
# Preprocess text 
# ---------------------------------------------------------

df = text_preproceccing(df, ["TEXT_ALL"])


In [49]:
df.shape

(4039906, 39)

In [50]:
# ---------------------------------------------------------
# TF‑IDF + SVD features
# ---------------------------------------------------------

df_svd, X_tfidf, tfidf_vectorizer, svd_model = build_tfidf_svd(df["TEXT_ALL"])
df = pd.concat([df, df_svd], axis=1)


In [52]:
# ---------------------------------------------------------
# Topic modelling (NMF)
# ---------------------------------------------------------
df_topics, nmf_model = build_nmf_topics(X_tfidf)
df = pd.concat([df, df_topics], axis=1)


In [53]:
df.shape

(4039906, 154)

In [55]:
# ---------------------------------------------------------
# Prepare labels for TEXT_RISK_SCORE
# ---------------------------------------------------------

risk_map = {
    "failed": "high",
    "low": "medium",
    "medium": "medium",
    "high": "low"
}

df["TEXT_RISK_LABEL"] = df["OFFERS_BIN"].map(risk_map)


In [56]:
# ---------------------------------------------------------
# Train SVM classifier
# ---------------------------------------------------------

df_text = df[df["TEXT_RISK_LABEL"].notna()].copy()

texts = df_text["TEXT_ALL"].fillna("").astype(str)
labels = df_text["TEXT_RISK_LABEL"].astype(str)

svm_model, tfidf_svm, label_encoder = build_text_risk_classifier(texts, labels)



In [57]:
# ---------------------------------------------------------
# Predict TEXT_RISK_SCORE
# ---------------------------------------------------------

df["TEXT_RISK_SCORE"] = predict_text_risk(
    df["TEXT_ALL"],
    svm_model,
    tfidf_svm,
    label_encoder
)



In [58]:
df.head()

,YEAR,ID_TYPE,XSD_VERSION,CANCELLED,CORRECTIONS,ISO_COUNTRY_CODE,CAE_TYPE,B_AWARDED_BY_CENTRAL_BODY,TYPE_OF_CONTRACT,TAL_LOCATION_NUTS,B_DYN_PURCH_SYST,ID_LOT,B_EU_FUNDS,TOP_TYPE,B_ACCELERATED,OUT_OF_DIRECTIVES,CRIT_CODE,CRIT_PRICE_WEIGHT,B_ELECTRONIC_AUCTION,NUMBER_AWARDS,B_AWARDED_TO_A_GROUP,WIN_COUNTRY_CODE,B_CONTRACTOR_SME,B_SUBCONTRACTED,TEXT_ALL,AWARD_QUARTER,DAYS_TO_AWARD,CPV_DIVISION,CPV_GROUP,CPV_CLASS,IS_FAILED_TENDER,IS_LOW_COMPETITION,OFFERS_BIN,VALUE_BIN,HAS_MULTIPLE_LOTS,LOTS_BIN,VALUE_EURO_MISSING,AWARD_VALUE_EURO_MISSING,NUMBER_OFFERS_MISSING,NLP_SVD_0,NLP_SVD_1,NLP_SVD_2,NLP_SVD_3,NLP_SVD_4,NLP_SVD_5,NLP_SVD_6,NLP_SVD_7,NLP_SVD_8,NLP_SVD_9,NLP_SVD_10,NLP_SVD_11,NLP_SVD_12,NLP_SVD_13,NLP_SVD_14,NLP_SVD_15,NLP_SVD_16,NLP_SVD_17,NLP_SVD_18,NLP_SVD_19,NLP_SVD_20,NLP_SVD_21,NLP_SVD_22,NLP_SVD_23,NLP_SVD_24,NLP_SVD_25,NLP_SVD_26,NLP_SVD_27,NLP_SVD_28,NLP_SVD_29,NLP_SVD_30,NLP_SVD_31,NLP_SVD_32,NLP_SVD_33,NLP_SVD_34,NLP_SVD_35,NLP_SVD_36,NLP_SVD_37,NLP_SVD_38,NLP_SVD_39,NLP_SVD_40,NLP_SVD_41,NLP_SVD_42,NLP_SVD_43,NLP_SVD_44,NLP_SVD_45,NLP_SVD_46,NLP_SVD_47,NLP_SVD_48,NLP_SVD_49,NLP_SVD_50,NLP_SVD_51,NLP_SVD_52,NLP_SVD_53,NLP_SVD_54,NLP_SVD_55,NLP_SVD_56,NLP_SVD_57,NLP_SVD_58,NLP_SVD_59,NLP_SVD_60,NLP_SVD_61,NLP_SVD_62,NLP_SVD_63,NLP_SVD_64,NLP_SVD_65,NLP_SVD_66,NLP_SVD_67,NLP_SVD_68,NLP_SVD_69,NLP_SVD_70,NLP_SVD_71,NLP_SVD_72,NLP_SVD_73,NLP_SVD_74,NLP_SVD_75,NLP_SVD_76,NLP_SVD_77,NLP_SVD_78,NLP_SVD_79,NLP_SVD_80,NLP_SVD_81,NLP_SVD_82,NLP_SVD_83,NLP_SVD_84,NLP_SVD_85,NLP_SVD_86,NLP_SVD_87,NLP_SVD_88,NLP_SVD_89,NLP_SVD_90,NLP_SVD_91,NLP_SVD_92,NLP_SVD_93,NLP_SVD_94,NLP_SVD_95,NLP_SVD_96,NLP_SVD_97,NLP_SVD_98,NLP_SVD_99,NLP_TOPIC_0,NLP_TOPIC_1,NLP_TOPIC_2,NLP_TOPIC_3,NLP_TOPIC_4,NLP_TOPIC_5,NLP_TOPIC_6,NLP_TOPIC_7,NLP_TOPIC_8,NLP_TOPIC_9,NLP_TOPIC_10,NLP_TOPIC_11,NLP_TOPIC_12,NLP_TOPIC_13,NLP_TOPIC_14,TEXT_RISK_LABEL,TEXT_RISK_SCORE
0,2008,3,D205,0,0,DE,8,Unknown,W,DED31,0,Unknown,Unknown,OPE,0,0,M,NaN,0,1,0,DE,0,Unknown,unknown preis qualit t 60 40,3.00,-73.00,45,452,4521,0,1,low,NaN,0,single,1,0,0,0.07,0.20,0.02,-0.25,0.09,-0.04,0.37,-0.32,-0.04,-0.00,-0.03,0.14,0.16,0.01,-0.02,0.03,-0.04,-0.09,0.09,-0.02,-0.02,0.02,0.14,-0.07,-0.18,-0.03,0.10,-0.11,0.06,-0.09,0.07,-0.03,-0.03,-0.07,0.02,0.00,-0.01,0.04,-0.01,-0.01,-0.01,-0.03,0.03,0.03,0.02,-0.06,-0.03,-0.05,-0.03,0.01,-0.01,0.04,-0.01,0.05,0.01,-0.03,0.04,0.05,-0.05,-0.02,-0.02,-0.03,0.02,0.05,0.02,-0.01,0.02,0.01,0.01,-0.04,-0.07,0.11,0.01,0.05,0.06,-0.11,0.07,0.08,-0.03,0.05,0.03,-0.03,-0.04,0.08,-0.04,0.04,-0.09,-0.11,0.08,0.00,-0.00,-0.03,0.02,-0.00,-0.01,0.02,0.01,0.06,0.02,-0.04,0.00,0.00,0.00,0.00,0.00,0.00,0.04,0.00,0.00,0.00,0.00,0.03,0.00,0.00,0.00,medium,low
1,2008,3,D205,0,0,DE,3,Unknown,W,DE913,0,Unknown,N,OPE,0,0,L,100.00,0,1,0,DE,0,N,unknown unknown unknown,4.00,-6.00,45,452,4521,0,0,medium,NaN,0,single,1,0,0,1.00,-0.01,-0.01,0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,-0.00,-0.00,0.00,-0.01,-0.00,-0.00,0.01,0.00,0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,0.00,-0.00,0.00,-0.00,-0.00,-0.00,-0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,-0.00,0.00,0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,-0.00,0.00,0.00,0.00,-0.00,0.03,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,medium,medium
2,2008,3,D205,0,0,FR,3,Unknown,W,Unknown,0,Unknown,N,OPE,0,0,M,NaN,0,1,0,FR,0,N,unknown prix d lai d intervention d urgence 80 20,4.00,-37.00,45,454,4542,1,0,failed,NaN,0,single,1,1,0,0.06,0.12,0.02,0.00,0.05,0.02,0.03,0.07,-0.00,-0.02,0.16,-0.08,-0.00,0.34,0.04,0.01,-0.04,-0.06,0.02,-0.09,-0.02,-0.03,-0.04,0.07,0.05,-0.05,-0.00,-0.03,0.09,0.02,-0.02,-0.01,-0.01,-0.01,-0.03,0.02,-0.01,-0.01,-0.04,0.08,0.01,-0.09,0.04,0.02,-0.09,-0.12,-0.03,0.04,0.07,0.02,0.03,0.00,0.07,0.07,0.05,-0.07,-0.00,-0.04,0.01,0.01,0.05,0.06,

In [59]:
df.columns

Index(['YEAR', 'ID_TYPE', 'XSD_VERSION', 'CANCELLED', 'CORRECTIONS',
       'ISO_COUNTRY_CODE', 'CAE_TYPE', 'B_AWARDED_BY_CENTRAL_BODY',
       'TYPE_OF_CONTRACT', 'TAL_LOCATION_NUTS',
       ...
       'NLP_TOPIC_7', 'NLP_TOPIC_8', 'NLP_TOPIC_9', 'NLP_TOPIC_10',
       'NLP_TOPIC_11', 'NLP_TOPIC_12', 'NLP_TOPIC_13', 'NLP_TOPIC_14',
       'TEXT_RISK_LABEL', 'TEXT_RISK_SCORE'],
      dtype='str', length=156)

In [60]:
# ---------------------------------------------------------
# SAVE TOPICS FOR GERMANY
# ---------------------------------------------------------

# Filter Germany
df_de = filter_germany(df)

# Select topic columns
topic_cols = [c for c in df_de.columns if c.startswith("TOPIC_")]

# Extract text + topics
df_de_topics = df_de[["TEXT_ALL"] + topic_cols].copy()

# Reset index for safety
df_de_topics = df_de_topics.reset_index(drop=True)

# Save topics
df_de_topics.to_pickle("../data/topics_de.pkl")

In [61]:
# drop unnecessary column
df = df.drop(columns="TEXT_ALL", errors="ignore").reset_index(drop=True)


#### Notes: NLP Pipeline for Tender Risk Prediction

1. Dataset Size After Features Engineering
- Full EU dataset:
  - Before: 4,039,906 rows × 41 columns
  - After: 4,039,906 rows × 154 columns

2. Text fields provide additional signals not captured by structured variables.  
Short titles and evaluation‑related text (TITLE, CRIT_CRITERIA, CRIT_WEIGHTS AS TEXT_ALL) reveal complexity, niche requirements, and multi‑criteria scoring patterns that strongly influence bidder participation and failure risk.

3. TF‑IDF offers a scalable and domain‑appropriate representation of tender text.  
It efficiently captures important terms and patterns without requiring heavy linguistic models, making it suitable for millions of records and classical ML workflows.

4. Dimensionality reduction (SVD) converts high‑dimensional TF‑IDF vectors into compact numerical features.  
This reduces sparsity, stabilizes downstream models, and enables seamless integration with structured predictors such as CPV, procedure type, and tender value.

5. Topic modelling (NMF) introduces interpretable thematic structure.  
Extracted topics highlight procurement areas with systematically higher failure rates, improving interpretability and analytical insight.

6. A text‑based classifier (TF‑IDF + SVM) produces a high‑level TEXT_RISK_SCORE.  
It learns patterns associated with failed, medium‑risk, and safe tenders based solely on text, generating a categorical risk signal usable even when raw text is unavailable.

7. TEXT_RISK_SCORE is essential for downstream applications such as the risk simulator.  
It allows the model to incorporate text‑derived risk information without requiring free‑text input, enabling scenario simulations based on a small set of structured parameters.

8. The combined pipeline remains interpretable, scalable, and robust.  
TF‑IDF + SVD ensures numerical stability, NMF adds thematic insight, and SVM provides a practical risk score — together forming a balanced NLP module that strengthens the overall tender risk prediction model.

---------
### SAVE DATASET

--------

In [62]:
df.to_pickle("../data/dataset_nlp.pkl")